# Word Error Rate (WER) Evaluation

In [ ]:
import re

In [ ]:
def normalize_text(text):    text = text.lower()    text = re.sub(r"[^a-z0-9' ]+", " ", text)    text = re.sub(r"\s+", " ", text).strip()    return text.split() if text else []

In [ ]:
def edit_distance_ops(ref_words, hyp_words):    n, m = len(ref_words), len(hyp_words)    dp = [[0] * (m + 1) for _ in range(n + 1)]    for i in range(n + 1):        dp[i][0] = i    for j in range(m + 1):        dp[0][j] = j    for i in range(1, n + 1):        for j in range(1, m + 1):            if ref_words[i - 1] == hyp_words[j - 1]:                dp[i][j] = dp[i - 1][j - 1]            else:                substitution = dp[i - 1][j - 1] + 1                insertion = dp[i][j - 1] + 1                deletion = dp[i - 1][j] + 1                dp[i][j] = min(substitution, insertion, deletion)    i, j = n, m    substitutions = insertions = deletions = 0    while i > 0 or j > 0:        if i > 0 and j > 0 and ref_words[i - 1] == hyp_words[j - 1]:            i -= 1            j -= 1        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:            substitutions += 1            i -= 1            j -= 1        elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:            insertions += 1            j -= 1        else:            deletions += 1            i -= 1    return substitutions, deletions, insertions, dp[n][m]

In [ ]:
def wer(reference, hypothesis):    ref_words = normalize_text(reference)    hyp_words = normalize_text(hypothesis)    substitutions, deletions, insertions, _ = edit_distance_ops(ref_words, hyp_words)    n = len(ref_words) if len(ref_words) > 0 else 1    return (substitutions + deletions + insertions) / n

In [ ]:
def batch_wer(pairs):    total_s = total_d = total_i = total_n = 0    per_example = []    for reference, hypothesis in pairs:        ref_words = normalize_text(reference)        hyp_words = normalize_text(hypothesis)        s, d, i, _ = edit_distance_ops(ref_words, hyp_words)        n = len(ref_words) if len(ref_words) > 0 else 1        example_wer = (s + d + i) / n        per_example.append(example_wer)        total_s += s        total_d += d        total_i += i        total_n += len(ref_words)    corpus_n = total_n if total_n > 0 else 1    corpus_wer = (total_s + total_d + total_i) / corpus_n    return {        "per_example_wer": per_example,        "corpus_wer": corpus_wer,        "substitutions": total_s,        "deletions": total_d,        "insertions": total_i,        "total_words": total_n    }

In [ ]:
pairs = [    ("the quick brown fox jumps over the lazy dog", "the quick brown fox jump over the lazy dog"),    ("he went to the market", "he went to market"),    ("speech recognition is hard", "speech recognition is hard"),    ("please call me tomorrow", "please call me tomorrow morning"),]results = batch_wer(pairs)for (reference, hypothesis), example_wer in zip(pairs, results["per_example_wer"]):    print(f"ref: {reference}")    print(f"hyp: {hypothesis}")    print(f"wer: {example_wer:.3f}")    print("-" * 40)print(f"corpus wer: {results['corpus_wer']:.3f}")print(f"substitutions: {results['substitutions']}")print(f"deletions: {results['deletions']}")print(f"insertions: {results['insertions']}")print(f"total reference words: {results['total_words']}")